# EDIS 8100 · Week 13 · Reading Research Critically, and Your Own AI Trace

### Reading a published paper's category, then building one about yourself

**Estimated time:** about 30 minutes for the core path.

Fall 2026 · Dr. Hakeoung Hannah Lee · University of Virginia

---

The reading hour asked two questions of three published papers: who does this research say the problem is inside, and how much of this paper did anybody check? This notebook asks both of them about you.

## How to work through this notebook

**Every code cell here runs as written.** You can press Shift and Enter from the top of this notebook to the bottom without editing anything, and nothing will break and nothing will be missing. That is tested before the notebook is published: a clean machine with no data on it runs the whole file end to end.

**The cells marked with a pencil already hold a working value.** Changing it and re-running shows you what that choice was doing to the result, which is the point of marking it. If you never touch one, you still get every finding this notebook is about.

**What is graded is what you write in the interpretation cells:** what the method can see, what it cannot, and what you would refuse to conclude from it.

**If you work in R, SPSS, or Stata, or have not written code before,** that is the expected starting point here. Read the code the way you read a methods section. Square brackets after a table, as in `df[df.words > 50]`, are subsetting. A line ending in `.mean()`, `.sum()`, or `.value_counts()` is an aggregation.

## Setup

**This is the only lab all semester whose data are not published.**

Every other week you analysed records somebody else collected and released under a licence: secondary students in Portugal, distance learners at the Open University, school writers in the United States, children with a robot in a Swiss lab, users of a tutoring app in Korea, forum posters in an open online course, undergraduates in a Spanish computer networks course, players of two science games. In none of those files did anyone consent to being your homework.

Tonight the file is yours. You generated it, you kept it, you were told in Week 1 that this session was coming, and nobody else opens it. That is the closest this course can come to answering, about itself, the question it has asked about everybody else's data for fourteen weeks.

**Either path works.**

- **Your own log.** Set `MY_LOG_PATH` below to a file you export from whatever assistant you used. Plain text is fine. Nobody sees it but you, and it is not collected.
- **A published transcript instead.** Leave `MY_LOG_PATH` empty and the notebook runs on `collab-chat/chat_logs.csv`: 1,374 real messages from eight groups of undergraduates working over four days in February 2021 in a computer networks course at Universidad de Valladolid, released **CC BY 4.0** by Cristina Villa-Torrano and colleagues. Every measure in Sections 2 and 3 works on it. Only Section 4 asks something that needs the trace to be your own.

Take either without asking. The notebook does not record which one you used.

In [ ]:
%matplotlib inline
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/HakeoungLee/edis8100-datasets/main"

OKABE = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
         "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#666666"}

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Your turn. Path to your own exported conversation. Leave "" to use the published transcript.
MY_LOG_PATH = ""

print("Setup complete.")
print("MY_LOG_PATH is", repr(MY_LOG_PATH), "->",
      "your own log" if MY_LOG_PATH else "the published transcript path")

---

## Section 1. Getting a conversation into a table

A conversation is a sequence of turns, each with an author and some text. That is all the structure the rest of this notebook needs, and it is worth noticing how little it is.

The parser below handles the format an exported assistant conversation usually arrives in: lines beginning with a speaker label such as `You:`, `Me:`, `User:`, `Assistant:`, `ChatGPT:`, or `Claude:`. If yours is not like that, paste it into a text file with `You:` and `Assistant:` at the start of each turn and it will parse. Counting twenty turns by hand is also a legitimate way to do this exercise; the arithmetic is not what matters here.

In [ ]:
ME_LABELS   = r"(?:you|me|user|human|q)"
THEM_LABELS = r"(?:assistant|chatgpt|gpt|claude|gemini|copilot|bot|a)"


def parse_labelled_text(text):
    """Split a transcript into turns using speaker labels at the start of a line."""
    pattern = re.compile(r"^\s*(" + ME_LABELS + r"|" + THEM_LABELS + r")\s*:\s*", re.I | re.M)
    parts = pattern.split(text)
    rows = []
    for label, body in zip(parts[1::2], parts[2::2]):
        who = "me" if re.fullmatch(ME_LABELS, label.strip(), re.I) else "them"
        body = body.strip()
        if body:
            rows.append({"speaker": who, "text": body})
    return pd.DataFrame(rows)


def load_published_transcript():
    """The path for anyone not using their own log: one real group's chat."""
    df = pd.read_csv(BASE + "/collab-chat/chat_logs.csv", sep=";", encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    df = df[df["Group"] == df["Group"].value_counts().idxmax()].copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
    busiest = df["authorid"].value_counts().idxmax()
    df["speaker"] = np.where(df["authorid"] == busiest, "me", "them")
    out = df.rename(columns={"message": "text"})[["speaker", "text", "timestamp"]]
    return out.reset_index(drop=True)


if MY_LOG_PATH:
    with open(MY_LOG_PATH, encoding="utf-8", errors="replace") as fh:
        turns = parse_labelled_text(fh.read())
    source = "your own log"
    if turns.empty:
        print("No labelled turns found in that file. Using the published transcript instead.")
        turns, source = load_published_transcript(), "the published transcript"
else:
    turns, source = load_published_transcript(), "the published transcript"

if "timestamp" not in turns.columns:
    turns["timestamp"] = pd.NaT

turns["text"] = turns["text"].astype(str)
turns["words"] = turns["text"].str.split().str.len()

print("Source:", source)
print(len(turns), "turns,", int(turns["words"].sum()), "words in total")
turns.head(6)[["speaker", "words", "text"]]

---

## Section 2. Turns, share, and latency

The three numbers are turn count, authorship share, and response latency. Each one is cheap to compute and each one is a decision about what counts.

Notice what the third needs that the first two do not: a clock. If your export dropped the timestamps, latency is simply unavailable to you, and that is worth registering rather than working around. It is the same situation as Week 6, where the corpus recorded nothing whatever about the children in it.

**If you are on the published-transcript path, look at the quartiles before you read anything into them.** They come out as 60 seconds, 60 seconds, 60 seconds. That file records time to the nearest minute, so every gap it can express is a multiple of 60 and the distribution you are looking at is the clock's, not the conversation's. Latency is not measurable in that file at the resolution the question needs. Say so rather than reporting a median.

In [ ]:
mine   = turns[turns.speaker == "me"].copy()
theirs = turns[turns.speaker == "them"].copy()
n_turns = len(turns)

share_turns = len(mine) / n_turns if n_turns else float("nan")
total_words = turns["words"].sum()
share_words = mine["words"].sum() / total_words if total_words else float("nan")

print("Turns:                 %d  (%d mine, %d theirs)" % (n_turns, len(mine), len(theirs)))
print("My share of turns:     %.1f%%" % (100 * share_turns))
print("My share of words:     %.1f%%" % (100 * share_words))
print("Median words per turn: mine %.0f, theirs %.0f"
      % (mine["words"].median(), theirs["words"].median()))

gaps = pd.Series(dtype=float)
if turns["timestamp"].notna().sum() > 2:
    ordered = turns.dropna(subset=["timestamp"]).sort_values("timestamp")
    g = ordered["timestamp"].diff().dt.total_seconds().dropna()
    gaps = g[g > 0]

if len(gaps):
    print("\nGap before the next message: median %.0f s, quartiles %.0f to %.0f s"
          % (gaps.median(), gaps.quantile(0.25), gaps.quantile(0.75)))
else:
    print("\nNo usable gaps. Latency is unavailable here, and that is a finding about the export "
          "rather than about the person.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].bar(["mine", "theirs"], [mine["words"].sum(), theirs["words"].sum()],
            color=[OKABE["blue"], OKABE["grey"]])
axes[0].set_title("Words written (" + source + ")")
axes[0].set_ylabel("words")

if len(gaps):
    axes[1].hist(np.log10(gaps.clip(lower=1)), bins=20, color=OKABE["orange"])
    axes[1].set_title("Gap before the next message")
    axes[1].set_xlabel("log10 seconds")
else:
    axes[1].text(0.5, 0.5, "no usable timestamps", ha="center", va="center",
                 transform=axes[1].transAxes, color=OKABE["grey"])
    axes[1].set_title("Gap before the next message")
    axes[1].set_xticks([])
    axes[1].set_yticks([])

plt.tight_layout()
plt.show()

---

## Section 3. Labelling your turns

The next cell does to you exactly what Yang and colleagues did to their participants, and what Kaliisa and colleagues did to a literature. It sorts your turns into four kinds using rules somebody else wrote, computes proportions, and prints a label of the sort a paper would use.

**Read this before you run it.** The label will probably be uncomfortable, and that is intended rather than accidental. It is also not a finding about you. It is the output of a keyword rule applied to text, on a sample of one, with no validation of any kind. It has exactly the evidential status of the categories you spent the 3:40 hour taking apart, which is why it is here.

The rules can be changed. Change them and the label changes, which is the second thing worth noticing.

**If you are on the published-transcript path, expect the rule to classify nothing at all.** It will report 100 percent unclassified, and that is not a bug in the notebook. Those undergraduates were working in Spanish; the four rules are English keywords. A category instrument works in the language it was built in, and applied to a conversation it was not built for it returns an empty result while looking exactly as authoritative as it did before.

Read that again before moving on, because it is the sharpest version of this week's argument that the notebook can produce. The instrument did not report that it could not classify this language. It returned zeros. Somebody reading only the output would conclude that these students never asked for anything.

**The second count is the reading hour's other question, asked about you.** After the first cell, a second one counts how many of your turns asked the model to justify something, name a source, or check itself. That number is not a virtue score and it is not graded. Plenty of legitimate uses of an assistant need no verification at all: fixing a syntax error, rephrasing a sentence, converting a date format. The number is only worth something next to what you were asking for at the time.

In [ ]:
# Your turn. These four rules construct the category. Edit them and the label moves.
RULES = {
    "asking for a fact":      r"\b(what is|who is|when did|define|meaning of|how many|which)\b",
    "asking for production":  r"\b(write|draft|generate|make|create|code|rewrite|summari[sz]e|translate)\b",
    "asking for critique":    r"\b(critique|why|is this right|check|wrong|disagree|weakness|problem with|flaw)\b",
    "asking for reassurance": r"\b(is that ok|does this make sense|am i|should i|good enough)\b",
}

NAMES = {"asking for a fact":      "a lookup-oriented user",
         "asking for production":  "a production-oriented user",
         "asking for critique":    "a critique-oriented user",
         "asking for reassurance": "a reassurance-seeking user"}


def classify(text):
    for kind, pattern in RULES.items():
        if re.search(pattern, text, re.I):
            return kind
    return "unclassified"


mine["kind"] = mine["text"].map(classify)
counts = mine["kind"].value_counts()
props = counts / counts.sum()

print("How my turns were sorted:\n")
for kind, n in counts.items():
    print("  %-24s %4d   %3.0f%%" % (kind, n, 100 * props[kind]))

named = props.drop(labels=["unclassified"], errors="ignore")
if len(named):
    top = named.idxmax()
    print("\nA paper would write: this participant is %s (%.0f%% of classified turns)."
          % (NAMES[top], 100 * named[top]))
else:
    print("\nNothing was classified. That is a result about the rule, not about the person.")

fig, ax = plt.subplots(figsize=(7, 3.2))
counts.sort_values().plot.barh(ax=ax, color=OKABE["purple"])
ax.set_title("Turns sorted by a rule somebody else wrote")
ax.set_xlabel("turns")
plt.tight_layout()
plt.show()


# Your turn. What counts as asking the model to justify, cite, or check itself.
VERIFY = r"\b(source|cite|citation|reference|evidence|how do you know|are you sure|double.?check|verify|is that (right|true|correct)|where does that come from|according to)\b"

mine["asks_for_backing"] = mine["text"].str.contains(VERIFY, case=False, regex=True)
k = int(mine["asks_for_backing"].sum())
print("\nTurns of mine that ask the model to justify, cite, or check: %d of %d (%.0f%%)"
      % (k, len(mine), 100 * k / max(len(mine), 1)))
if k == 0:
    print("Zero is a common result and it is a fact about how the tool got used, not about you.")
print("Hold this next to the first count. A run of production requests with no backing requests\n"
      "is a different picture from a run of fact requests with no backing requests.")

---

## Section 4. Argue against it

This is the deliverable of the session and it is not collected.

Write three things the label gets wrong. For each one, name the evidence that would settle it, and say whether that evidence is anywhere in the trace. Almost none of it will be.

Then answer the evening's main question, in writing rather than in your head:

> The students in Yang and colleagues (2022) knew all of this about themselves too. Nobody asked them. **What follows for a study you are about to run?**

If you are on the published-transcript path, write it about the busiest author in that group, and say explicitly what changes when the person you are describing is somebody you have never met.

### Your answer

**Three things the label gets wrong, and what would settle each:**

1.
2.
3.

**What follows for my own study:**


---

## Before December 2

Your project has a category in it. Every project does: a group you compare, a population you bound, a threshold you pick, a set of studies you pool.

Before you present, be able to say in one sentence each where the category came from, what in your argument would collapse without it, and what two people it would treat as the same.

That last one is the move you have made every week since Week 3, and it is the one a committee will ask you about.